# 01 — Exploratory Data Analysis
Vue d'ensemble de tes données de games. À lancer après le premier backfill.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pipeline.load_db import get_games_as_df

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

df = get_games_as_df()
print(f'Total games: {len(df)}')
df.head()

In [ ]:
# Global winrate
wr = df['win'].mean()
print(f'Overall winrate: {wr:.1%} over {len(df)} games')

# Winrate by champion
champ_stats = df.groupby('champion_name').agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
    avg_kda=('kda_ratio', 'mean'),
    avg_cs=('cs_per_min', 'mean'),
    avg_kp=('kill_participation', 'mean'),
).round(3).sort_values('games', ascending=False)

champ_stats

In [ ]:
# Winrate by champion (bar chart)
fig, ax = plt.subplots(figsize=(10, 5))
data = champ_stats[champ_stats['games'] >= 3].sort_values('winrate', ascending=True)
colors = ['#27AE60' if wr >= 0.5 else '#E74C3C' for wr in data['winrate']]
bars = ax.barh(data.index, data['winrate'], color=colors)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Winrate')
ax.set_title('Winrate par champion (min. 3 games)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# Annotate with game count
for bar, (_, row) in zip(bars, data.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['games']}g", va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Rolling winrate over time (last 30 games)
df_sorted = df.sort_values('game_date')
df_sorted['rolling_wr'] = df_sorted['win'].rolling(window=10, min_periods=3).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_sorted['game_date'], df_sorted['rolling_wr'], color='#2E86AB', lw=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(df_sorted['game_date'], df_sorted['rolling_wr'], 0.5,
                where=df_sorted['rolling_wr'] >= 0.5, alpha=0.2, color='#27AE60')
ax.fill_between(df_sorted['game_date'], df_sorted['rolling_wr'], 0.5,
                where=df_sorted['rolling_wr'] < 0.5, alpha=0.2, color='#E74C3C')
ax.set_ylabel('Winrate (rolling 10g)')
ax.set_title('Évolution du winrate dans le temps')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
# CS/min distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, champ in zip(axes, df['champion_name'].value_counts().head(2).index):
    subset = df[df['champion_name'] == champ]['cs_per_min'].dropna()
    ax.hist(subset, bins=15, color='#2E86AB', edgecolor='white')
    ax.axvline(subset.mean(), color='#E74C3C', linestyle='--', label=f'Moy: {subset.mean():.1f}')
    ax.set_title(f'CS/min — {champ}')
    ax.set_xlabel('CS/min')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation: mental state vs winrate
mental_wr = df.groupby('mental_pregame').agg(
    games=('win', 'count'),
    winrate=('win', 'mean')
).dropna()

if not mental_wr.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#27AE60']
    bars = ax.bar(mental_wr.index.astype(int), mental_wr['winrate'],
                  color=[colors[int(i)-1] for i in mental_wr.index])
    ax.axhline(0.5, color='gray', linestyle='--')
    ax.set_xlabel('Mental pré-game (1=mauvais, 5=excellent)')
    ax.set_ylabel('Winrate')
    ax.set_title('Corrélation mental pré-game / winrate')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    
    for bar, (idx, row) in zip(bars, mental_wr.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{row['games']}g", ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Pas encore de données mental_pregame — remplis quelques games d\'abord.')

In [ ]:
# Loss causes breakdown
loss_causes = df[~df['win']]['loss_cause'].value_counts().dropna()
if not loss_causes.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    loss_causes.plot(kind='barh', ax=ax, color='#E74C3C')
    ax.set_title('Causes de défaite')
    plt.tight_layout()
    plt.show()
else:
    print('Pas encore de causes de défaite renseignées.')